In [ ]:
import pyodbc
import pandas as pd
import ast

In [ ]:
df = pd.read_csv("../data/all_students.csv")

# Group and convert np.int64 to a  int
result = (
    df.groupby("EMAIL")["SUBGROEPID"]
      .unique()
      .reset_index()
)
result["SUBGROEPID"] = result["SUBGROEPID"].apply(lambda x: list(map(int, x)))
result.head()
result.to_csv("../data/cleaned/students/users_with_subgroeps.csv", index=False)


In [ ]:
df = result

# (remove whitespace)
df["EMAIL"] = df["EMAIL"].str.strip()

server = "127.0.0.1,1500"
database = "DEP2_staging"
username = "sa"
password = "dep2025-G12"
driver = "ODBC Driver 17 for SQL Server"

conn_str = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password}"
)
conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

insert_query = """
IF NOT EXISTS (SELECT 1 FROM dbo.DimUser WHERE UserName = ?)
BEGIN
    INSERT INTO dbo.DimUser (UserName) VALUES (?);
END
"""

for email in df["EMAIL"]:
    cursor.execute(insert_query, email, email)


In [ ]:

conn.commit()
print("✅ Data inserted successfully.")


✅ New users added to DimUser successfully (existing ones skipped).


In [ ]:

df = pd.read_csv("../data/cleaned/students/users_with_subgroeps.csv")
df["EMAIL"] = df["EMAIL"].str.strip()

dimuser = pd.read_sql("SELECT UserKey, UserName FROM dbo.DimUser", conn)

#Merge: CSV EMAIL -> DimUser's UserName 
merged = df.merge(dimuser, left_on="EMAIL", right_on="UserName", how="left")

merged = merged.dropna(subset=["UserKey"])


# merged.to_csv("../data/emails_with_userkeys.csv", index=False) ter controle

C:\Users\Korneel\AppData\Local\Temp\ipykernel_28388\1659630374.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dimuser = pd.read_sql("SELECT UserKey, UserName FROM dbo.DimUser", conn)


In [ ]:
rows = []
for _, row in merged.iterrows():
    subgroup_list = ast.literal_eval(row["SUBGROEPID"])
    for subgroup_id in subgroup_list:
        rows.append((int(row["UserKey"]), int(subgroup_id)))

bridge_df = pd.DataFrame(rows, columns=["UserKey", "SubgroupKey"])

bridge_df = bridge_df.drop_duplicates()

cursor = conn.cursor()

insert_query = """
IF NOT EXISTS (
    SELECT 1 FROM dbo.BridgeUserSubgroup WHERE UserKey = ? AND SubgroupKey = ?
)
BEGIN
    INSERT INTO dbo.BridgeUserSubgroup (UserKey, SubgroupKey)
    VALUES (?, ?)
END
"""

for row in bridge_df.itertuples(index=False):
    cursor.execute(insert_query, row.UserKey, row.SubgroupKey, row.UserKey, row.SubgroupKey)

conn.commit()
cursor.close()
conn.close()
print("✅ Insert complete!")

✅ Insert complete!
